# Point-OCR 全流程（AutoDL / Unsloth）

按顺序执行每个代码块。本 notebook 覆盖：

1. 环境与路径  
2. 合成数据（HTML → 准星监督对）  
3. **train / val / test** 划分  
4. Unsloth Stage A 训练  
5. 验证集指标 + **测试集 5 例可视化**（图 + 模型输出）  
6. （可选）导出提示  

详细命令备忘：`docs/AUTODL.md` · Unsloth：`docs/UNSLOTH.md`

> 需要 **Linux + NVIDIA GPU** 才能训练/推理；造数据在 CPU 上也可跑（需 Playwright Chromium）。


## 0. 定位仓库根目录

后面所有相对路径都相对于仓库根。若在 `notebooks/` 下打开本文件，会自动 `chdir` 到上一级。


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("ROOT =", ROOT)
print("cwd  =", Path.cwd())


## 0.1 AutoDL 镜像（弱网必做）

先配置 PyPI / HuggingFace / Playwright 镜像再装包。

In [ ]:
import os
env = {
  'UV_INDEX_URL': 'https://mirrors.aliyun.com/pypi/simple',
  'UV_EXTRA_INDEX_URL': 'https://pypi.tuna.tsinghua.edu.cn/simple',
  'PIP_INDEX_URL': 'https://mirrors.aliyun.com/pypi/simple',
  'HF_ENDPOINT': 'https://hf-mirror.com',
  'PLAYWRIGHT_DOWNLOAD_HOST': 'https://npmmirror.com/mirrors/playwright',
  'GITHUB_PROXY': 'https://ghproxy.net',
}
os.environ.update(env)
for k,v in env.items():
    print(f'{k}={v}')


## 1. 环境检查

- `uv` 管理依赖；训练前需 `uv pip install unsloth`（CUDA 主机）  
- Chromium **只用于** HTML 渲染造数据，不是训练框架


In [ ]:
!uv --version
!nvidia-smi -L || echo "(无 GPU：仍可造数据；训练/可视化推理请换 CUDA 机器)"


In [ ]:
# 首次环境（按需取消注释）
# !uv sync --extra dev --extra synth
# !uv run playwright install chromium
# !uv sync --extra train
# !uv pip install unsloth
# !uv pip install jupyter ipykernel matplotlib
print("环境步骤按需执行")


## 2. 生成合成监督数据

管线：模板 HTML（`data-block-id`）→ Playwright 截图 → **文字墨迹 bbox** → 在块内做**角落/边缘分层采样**画准星 → ShareGPT JSONL。

准星采样已改为覆盖四角与左右端（适配横向长、纵向窄的文本行），不再挤在正中心。


In [ ]:
# 造数（首次或改模板后重跑；可加 --noise）
!uv run python data/scripts/build_synth_batch.py \
  --out data/processed/synth \
  --seed 0 \
  --r-min 3 --r-max 5 \
  --negatives 4

from point_ocr.dataset_format import load_jsonl
rows = load_jsonl(ROOT / "data/processed/synth/point_sharegpt.jsonl")
print("total pairs =", len(rows))
# 看一眼 region 分布
from collections import Counter
regs = Counter((r.get("metadata") or {}).get("region") for r in rows if not (r.get("metadata") or {}).get("is_negative"))
print("positive region counts:", dict(regs.most_common(12)))


## 3. 划分 train / val / test

默认 **90% / 5% / 5%**，打乱后切分，写出：

- `data/splits/train.jsonl` — 给 Unsloth 训练  
- `data/splits/val.jsonl` — 指标监控  
- `data/splits/test.jsonl` — 最终目视 + 指标（训练不可见）


In [ ]:
!uv run python data/scripts/split_train_val_test.py \
  --input data/processed/synth/point_sharegpt.jsonl \
  --out-dir data/splits \
  --train-frac 0.90 --val-frac 0.05 --test-frac 0.05 \
  --seed 42

print((ROOT / "data/splits/split_meta.json").read_text())


## 4. Unsloth 冒烟加载基座

确认 `ATH-MaaS/OvisOCR2` 能被 `FastVisionModel` 加载。失败见 `docs/UNSLOTH.md`。


In [ ]:
# !uv run python train/unsloth_stage_a.py --smoke-load-only --model ATH-MaaS/OvisOCR2
print("取消注释上一行做兼容性检查（需 GPU + unsloth）")


## 5. Stage A 训练（POINT QLoRA）

默认冻 vision。建议用 `tmux` 跑全量；下面先给一个可打断的小步调试命令。


In [ ]:
# 调试小跑（可先跑通）
# !uv run python train/unsloth_stage_a.py \
#   --data data/splits/train.jsonl \
#   --out checkpoints/stage_a_point_unsloth \
#   --max-samples 128 --max-steps 50

# 全量训练：
# !uv run python train/unsloth_stage_a.py \
#   --data data/splits/train.jsonl \
#   --out checkpoints/stage_a_point_unsloth

print("取消注释以开训；产出 checkpoints/stage_a_point_unsloth/lora_adapter")


## 6. 验证集快速打分（可选）

用仓库指标脚本对 val 的「预测 JSONL」打分。若还没有模型预测，可先跳过，直接做第 7 步目视。

完整离线评测也可：`eval/run_eval.py`（vLLM 后端）。


In [ ]:
from point_ocr.metrics import EvalExample, evaluate_examples, looks_like_over_extraction
print("metrics ready:", evaluate_examples([
    EvalExample("demo", "hello", "hello", False)
]).to_dict())


## 7. 测试集目视：抽出 5 张，成对显示「带准星图 | GT / 模型输出」

- **无微调权重时**：只显示 GT，并用「基座零样本」推理（若已装 unsloth/vLLM）  
- **有 LoRA 时**：加载 adapter 推理  

请先设 `ADAPTER_DIR`；若为空则只展示图 + GT。


In [ ]:
from IPython.display import display, Markdown
from PIL import Image
import matplotlib.pyplot as plt
from point_ocr.viz_test import pick_test_examples, row_image_and_target
from point_ocr.prompts import POINT_PROMPT

TEST_JSONL = ROOT / "data/splits/test.jsonl"
ADAPTER_DIR = ROOT / "checkpoints/stage_a_point_unsloth/lora_adapter"
BASE_MODEL = "ATH-MaaS/OvisOCR2"
N_SHOW = 5
SEED = 0
# True = 跑推理；False = 只看图+GT（无需 GPU）
RUN_INFER = ADAPTER_DIR.exists()

examples = pick_test_examples(TEST_JSONL, n=N_SHOW, seed=SEED, prefer_positive=True)
print(f"picked {len(examples)} | RUN_INFER={RUN_INFER} | adapter_exists={ADAPTER_DIR.exists()}")

_POINT_MODEL = None
_POINT_TOK = None

def predict_unsloth(image: Image.Image, prompt: str) -> str:
    global _POINT_MODEL, _POINT_TOK
    if _POINT_MODEL is None:
        from unsloth import FastVisionModel
        model, tok = FastVisionModel.from_pretrained(
            BASE_MODEL, load_in_4bit=True, use_gradient_checkpointing="unsloth"
        )
        if ADAPTER_DIR.exists():
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
            print("loaded adapter", ADAPTER_DIR)
        else:
            print("using base model (zero-shot)")
        FastVisionModel.for_inference(model)
        _POINT_MODEL, _POINT_TOK = model, tok

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    input_text = _POINT_TOK.apply_chat_template(messages, add_generation_prompt=True)
    inputs = _POINT_TOK(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out_ids = _POINT_MODEL.generate(**inputs, max_new_tokens=512, use_cache=True, do_sample=False)
    # strip prompt tokens
    gen = out_ids[0][inputs["input_ids"].shape[-1]:]
    return _POINT_TOK.decode(gen, skip_special_tokens=True).strip()

for i, row in enumerate(examples):
    img_path, gt, sid = row_image_and_target(row)
    img = Image.open(img_path).convert("RGB")
    pred = ""
    if RUN_INFER:
        try:
            pred = predict_unsloth(img, POINT_PROMPT)
        except Exception as e:
            pred = f"[infer error] {e}"
    else:
        pred = "(未推理：将 RUN_INFER=True 或先完成训练以加载 adapter)"

    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"[{i+1}/{len(examples)}] {sid}", fontsize=10)
    plt.show()
    display(Markdown(f"**GT**\n```\n{gt if gt else '∅'}\n```\n**PRED**\n```\n{pred}\n```"))


## 8. （可选）合并 LoRA 并导出 GGUF

见 `export/README.md` 与 `docs/AUTODL.md` 第 6 节。


In [ ]:
# !uv run python export/merge_lora.py \
#   --base ATH-MaaS/OvisOCR2 \
#   --adapter checkpoints/stage_a_point_unsloth/lora_adapter \
#   --out exports/OvisOCR2-Point-hf
print("导出按需取消注释")


## 验收提醒

- 准星应落在目标文字上（含角落/行首行尾），预测应接近 GT 单块  
- `over-extraction`（整页倾倒）应明显低于零样本基座  
- 负例（准星在空白/任务栏）应接近空串
